# caliper selftest on Colab

Runs `caliper selftest --full` on this GPU box and saves the Appendix-E
report as an artifact. Once the on-device oracle runner lands, an A100 without `nsys` is
**PASS** with `coverage: reduced` and
`not_validated: [clock_lock, ncu_crosscheck, powercap_throttle]`. Until then a
device-present run is `ERROR` (every oracle skipped) -- that is expected.

Commit `selftest-report.json` alongside the acceptance report.

In [ ]:
# bootstrap (skip if the dev.ipynb bootstrap already ran this session)
!curl -sSf https://sh.rustup.rs | sh -s -- -y >/dev/null && source $HOME/.cargo/env
!apt-get -qq install -y nsight-systems-cli 2>/dev/null || true
%cd /content
![ -d caliper ] || git clone --depth 1 https://github.com/mansoor-mamnoon/caliper
%cd caliper
!git pull --ff-only
!pip -q install -e ".[dev]"


In [ ]:
# the self-test; exit code is 0 PASS / 1 FAIL / 2 ERROR
import json, subprocess, sys
p = subprocess.run(["caliper", "selftest", "--full", "--json"], capture_output=True, text=True)
open("selftest-report.json", "w").write(p.stdout)
report = json.loads(p.stdout)
print("result:  ", report["result"])
print("coverage:", report["coverage"])
print("exit:    ", p.returncode)
for c in report["checks"]:
    print(f"  {c['status']:5}  {c['name']:24} {c['detail']}")
print("not validated:", ", ".join(report["not_validated"]))
# 0 PASS / 1 FAIL / 2 ERROR -- fail the cell on anything but PASS
sys.exit(p.returncode)


In [ ]:
# structural check: the committed report must validate
from caliper import _core
import json
report = json.load(open("selftest-report.json"))
report.pop("exit_code", None)
problems = _core.validate_selftest_json(json.dumps(report))
assert problems == [], problems
print("selftest-report.json validates")
